Animation of Rice growing areas in Sacramento Valley, California

In [1]:



#import libraries

import os
import zipfile
from glob import glob
from io import BytesIO

import geopandas as gpd
import rasterio
from rasterio.io import MemoryFile
from rasterio.mask import mask
from rasterio.plot import plotting_extent
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import imageio.v2 as imageio
from matplotlib.colors import ListedColormap



In [2]:

# File paths

shape_files = "/group/moniergrp/dbaral/run_project/input_data/shape_files"
crop_cdl_zips_path = "/group/moniergrp/CDL/zipped-original-files"
output_frames = "/group/moniergrp/dbaral/run_project/output_data/output_frame"
save_path = "/group/moniergrp/dbaral/run_project/output_data/animation"

os.makedirs(output_frames, exist_ok=True)
os.makedirs(save_path, exist_ok=True)



In [3]:

# Load county shapefile

counties = gpd.read_file(os.path.join(shape_files, "CA counties.shp"))

rice_names = [
    "Butte", "Colusa", "Glenn", "Yolo", "Yuba",
    "Placer", "Sacramento", "San Joaquin", "Sutter"
]

rice_counties = counties[counties["NAME"].isin(rice_names)].copy()



In [4]:

# Find CDL zip files

zip_files = sorted(glob(os.path.join(crop_cdl_zips_path, "*_cdls.zip")))

rice_code = 3
frame_files = []

#  colors
cmap = ListedColormap(["#f7f7f7", "#2ca25f"])


# Loop through zip files

for zf_path in zip_files:
    year = os.path.basename(zf_path).split("_")[0]

    with zipfile.ZipFile(zf_path, "r") as zip_ref:
        members = zip_ref.namelist()

        # keep only true tif files, skip folders or weird entries
        tif_list = [
            f for f in members
            if f.lower().endswith(".tif") and not f.endswith("/")
        ]

        if len(tif_list) == 0:
            print(f"Skipping {year}: no tif found")
            continue

        tif_name = tif_list[0]
        print(f"Processing {year}: {tif_name}")

        # read tif bytes directly from the zip
        with zip_ref.open(tif_name) as tif_bytes:
            tif_data = tif_bytes.read()

    # read raster from memory
    with MemoryFile(tif_data) as memfile:
        with memfile.open() as src:
            rice_counties_proj = rice_counties.to_crs(src.crs)
            shapes = list(rice_counties_proj.geometry)

            out_image, out_transform = mask(src, shapes, crop=True)
            out_image = out_image[0]

            rice_mask = np.where(out_image == rice_code, 1, 0)

            if src.nodata is not None:
                rice_mask = np.where(out_image == src.nodata, np.nan, rice_mask)

            extent = plotting_extent(out_image, out_transform)

 
    # Plot frame

    fig, ax = plt.subplots(figsize=(7.5, 7.5))

    ax.imshow(
        rice_mask,
        cmap=cmap,
        extent=extent,
        interpolation="none",
        origin="upper",
        zorder=1
    )

    rice_counties_proj.boundary.plot(
        ax=ax,
        color="black",
        linewidth=0.8,
        zorder=2
    )

    # optional labels
    for _, row in rice_counties_proj.iterrows():
        pt = row.geometry.representative_point()
        ax.text(
            pt.x,
            pt.y,
            row["NAME"],
            fontsize=7,
            ha="center",
            va="center",
            color="black",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.65, pad=1),
            zorder=3
        )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(
        f"Rice-Growing Area in California Rice Counties ({year})",
        fontsize=14,
        fontweight="bold",
        pad=12
    )

    rice_patch = mpatches.Patch(color="#2ca25f", label="Rice area")
    county_line = plt.Line2D([0], [0], color="black", lw=1, label="County boundary")
    ax.legend(
        handles=[rice_patch, county_line],
        loc="lower left",
        frameon=True,
        framealpha=0.95,
        fontsize=9
    )

    frame_file = os.path.join(output_frames, f"rice_{year}.png")
    plt.savefig(frame_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    frame_files.append(frame_file)



Processing 2008: 2008_30m_cdls.tif
Processing 2009: 2009_30m_cdls.tif
Processing 2010: 2010_30m_cdls.tif
Processing 2011: 2011_30m_cdls.tif
Processing 2012: 2012_30m_cdls.tif
Processing 2013: 2013_30m_cdls.tif
Processing 2014: 2014_30m_cdls.tif
Processing 2015: 2015_30m_cdls.tif
Processing 2016: 2016_30m_cdls.tif
Processing 2017: 2017_30m_cdls.tif
Processing 2018: 2018_30m_cdls.tif
Processing 2019: 2019_30m_cdls.tif
Processing 2020: 2020_30m_cdls.tif
Processing 2021: 2021_30m_cdls.tif
Processing 2022: 2022_30m_cdls.tif
Processing 2023: 2023_30m_cdls.tif


In [5]:

# Build GIF

gif_file = os.path.join(save_path, "rice_area_change_animation.gif")

images = [imageio.imread(f) for f in sorted(frame_files)]
imageio.mimsave(gif_file, images, duration=0.8, loop=0)

print("GIF saved to:", gif_file)

GIF saved to: /group/moniergrp/dbaral/run_project/output_data/animation/rice_area_change_animation.gif
